# 传统CV算法--在NN之前的江湖

- 图像滤波：通过局部像素加权运算来平滑图像、增强特征或消除噪声。
- 边缘检测：识别图像中灰度或颜色发生急剧变化的位置以提取物体轮廓。
- 图像分割：根据相似性（颜色、纹理等）将图像划分成若干有意义的区域或对象。

# 图像滤波1 -- 最简单就是均值滤波，所有像素点平等对待

In [ ]:
# 本代码展示如何向图像中添加椒盐噪声，并使用均值滤波器进行噪声去除

# 导入必要的库
import cv2  # OpenCV库，用于图像处理
import numpy as np  # NumPy库，用于数值计算
import matplotlib.pyplot as plt  # Matplotlib库，用于图像显示

# 使用OpenCV读取图像
image = cv2.imread("panda.png")  # 'panda.png'是图像文件的路径


# 这段代码定义了一个名为 add_salt_and_pepper_noise 的函数，用于向图像中添加椒盐噪声。
# 椒盐噪声是一种图像噪声，其中一些随机像素会被设置为最亮（通常是白色）或最暗（通常是黑色）
# 定义添加椒盐噪声的函数
def add_salt_and_pepper_noise(image, salt_prob, pepper_prob):
    """
    向图像中添加椒盐噪声。

    参数:
    image (numpy.ndarray): 原始图像。
    salt_prob (float): 添加白色（盐）噪声的概率。
    pepper_prob (float): 添加黑色（椒）噪声的概率。

    返回:
    numpy.ndarray: 添加椒盐噪声后的图像。
    """
    noisy_image = image.copy()  # 创建图像的副本以避免修改原始图像
    total_pixels = image.size  # 计算图像中的总像素数

    # 添加椒盐噪声（黑色）
    num_salt = np.ceil(salt_prob * total_pixels)  # 根据椒盐噪声声概率计算需要添加的椒盐噪声声像素数
    salt_coords = [
        np.random.randint(0, i - 1, int(num_salt)) for i in image.shape
    ]  # 随机生成椒盐噪声的坐标
    noisy_image[salt_coords[0], salt_coords[1], :] = 255  # 将椒盐噪声像素设置为白色

    # 添加椒盐噪声（白色）
    num_pepper = np.ceil(pepper_prob * total_pixels)  # 根据椒盐噪声概率计算需要添加的椒盐噪声像素数
    pepper_coords = [
        np.random.randint(0, i - 1, int(num_pepper)) for i in image.shape
    ]  # 随机生成椒盐噪声的坐标
    noisy_image[pepper_coords[0], pepper_coords[1], :] = 0  # 将椒盐噪声像素设置为黑色

    return noisy_image  # 返回添加了噪声的图像




In [ ]:
# 向图像添加椒盐噪声
# 设置椒噪声和盐噪声的概率为 2%
salt_and_pepper_image = add_salt_and_pepper_noise(
    image, salt_prob=0.02, pepper_prob=0.02
)

# 定义均值滤波器的核大小
# 设置核大小为 8x8
kernel_size = (8, 8)

# 使用均值滤波器对图像进行滤波
# 应用均值滤波以减少图像中的噪声
filtered_image = cv2.blur(salt_and_pepper_image, kernel_size)

# 使用 Matplotlib 显示原始图像、添加椒盐噪声后的图像和滤波后的图像
fig, axes = plt.subplots(1, 3, figsize=(15, 5))  # 设置绘图布局为 1 行 3 列

# 显示原始图像
axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))  # 将 BGR 格式转换为 RGB 格式
axes[0].set_title("Original Image")  # 设置图像标题
axes[0].axis("off")  # 关闭坐标轴显示

# 显示添加椒盐噪声后的图像
axes[1].imshow(cv2.cvtColor(salt_and_pepper_image, cv2.COLOR_BGR2RGB))  # 显示添加噪声后的图像
axes[1].set_title("Image with Salt and Pepper Noise")  # 设置图像标题
axes[1].axis("off")  # 关闭坐标轴显示

# 显示应用均值滤波后的图像
axes[2].imshow(cv2.cvtColor(filtered_image, cv2.COLOR_BGR2RGB))  # 显示滤波后的图像
axes[2].set_title("Filtered Image")  # 设置图像标题
axes[2].axis("off")  # 关闭坐标轴显示

plt.show()  # 展示所有图像

# 图像滤波1 -- 进阶一点：高斯滤波均值滤波，假设任何像素点周边噪声分布和其距离呈高斯分布

In [ ]:
# 这段代码展示了如何向图像中添加高斯噪声，并使用高斯滤波进行噪声去除。
# 代码首先读取原始图像，然后添加高斯噪声，并应用高斯滤波。
# 最后，使用 matplotlib 展示原始图像、添加噪声后的图像和去噪后的图像。


# 这段代码定义了一个名为 add_gaussian_noise 的函数，它用于向图像中添加高斯噪声。
# 高斯噪声是一种常见的图像噪声，它对图像的每个像素添加了根据高斯分布生成的随机值。
def add_gaussian_noise(image, mean=0, sigma=25):
    """
    向图像中添加高斯噪声。

    参数:
    image (numpy.ndarray): 原始图像。
    mean (float): 高斯噪声的均值，默认为 0。
    sigma (float): 高斯噪声的标准差，默认为 25。

    返回:
    numpy.ndarray: 添加高斯噪声后的图像。
    """
    # 获取图像的维度和通道数
    row, col, ch = image.shape

    # 生成与图像尺寸相同的高斯噪声
    gauss = np.random.normal(mean, sigma, (row, col, ch))

    # 将高斯噪声添加到原始图像上
    # 使用 np.clip 确保添加噪声后的像素值仍然在 0 到 255 的范围内
    noisy = np.clip(image + gauss, 0, 255)

    # 返回添加噪声后的图像，并将数据类型转换为无符号8位整型
    return noisy.astype(np.uint8)




In [ ]:
# 读取图像文件
original_image = image

# 向原始图像添加高斯噪声
noisy_image = add_gaussian_noise(original_image)

# 使用高斯滤波器对添加噪声的图像进行去噪处理
# 设置高斯滤波器的核大小为 5x5
denoised_image = cv2.GaussianBlur(noisy_image, (5, 5), 0)

# 使用 matplotlib 显示原始图像、带噪声的图像和去噪后的图像
plt.subplot(131)  # 设置显示位置
plt.imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))  # 显示原始图像（转换颜色空间为RGB）
plt.title("Original Image")  # 设置图像标题

plt.subplot(132)  # 设置显示位置
plt.imshow(cv2.cvtColor(noisy_image, cv2.COLOR_BGR2RGB))  # 显示带噪声的图像
plt.title("Noisy Image")  # 设置图像标题

plt.subplot(133)  # 设置显示位置
plt.imshow(cv2.cvtColor(denoised_image, cv2.COLOR_BGR2RGB))  # 显示去噪后的图像
plt.title("Denoised Image")  # 设置图像标题

# 显示所有图像
plt.show()

# 均值滤波 vs 高斯滤波

均值滤波适合对速度要求高、对质量要求不高的简单应用，而高斯滤波是更通用、效果更好的选择，特别适用于需要保持图像特征的高质量图像处理任务。在实际应用中，高斯滤波通常是首选的平滑滤波器，除非有严格的实时性要求。
## 图像处理中均值滤波与高斯滤波对比

| 对比维度 | 均值滤波 | 高斯滤波 |
|---------|----------|----------|
| **基本原理** | 邻域内所有像素的算术平均值 | 邻域内像素按高斯分布加权平均值 |
| **数学公式** | \( g(x,y) = \frac{1}{N} \sum f(i,j) \) | \( g(x,y) = \sum w(i,j)·f(i,j) \)，w为高斯权重 |
| **权重分布** | 均匀权重 | 钟形权重（中心大，四周小） |
| **平滑效果** | 均匀平滑，整体模糊 | 自然平滑，中心加权 |
| **边缘保持** | 差（边缘严重模糊） | 较好（边缘相对清晰） |
| **噪声消除** | 对高斯噪声一般，对脉冲噪声差 | 对高斯噪声效果显著，对脉冲噪声有一定效果 |
| **计算复杂度** | 低（权重相同，可优化） | 较高（需要计算权重） |
| **频率响应** | sinc函数形状，有旁瓣 | 高斯函数形状，无旁瓣 |
| **振铃效应** | 明显 | 无 |
| **参数调节** | 仅核大小 | 核大小和标准差σ |
| **主要应用** | 快速去噪、实时系统 | 精细图像处理、计算机视觉预处理 |
| **典型核示例** | 3×3: <br>`[[1/9, 1/9, 1/9],`<br>` [1/9, 1/9, 1/9],`<br>` [1/9, 1/9, 1/9]]` | 3×3(σ=1.0): <br>`[[0.075, 0.124, 0.075],`<br>` [0.124, 0.204, 0.124],`<br>` [0.075, 0.124, 0.075]]` |

## 2D、3D高斯滤波核（kernel）展示

In [ ]:
# 这段代码展示了如何生成一个高斯滤波器（高斯核），并使用 matplotlib 绘制其二维和三维图像。
# 高斯滤波器是一种常用于图像处理中的平滑滤波器，它根据高斯函数生成一个核，该核在图像处理中用于模糊图像和去除噪声。

# 导入所需的库
import numpy as np  # NumPy库，用于数值计算
import matplotlib.pyplot as plt  # Matplotlib库，用于图像显示
from mpl_toolkits.mplot3d import Axes3D  # Matplotlib的3D绘图工具包
from scipy.stats import multivariate_normal  # SciPy的多元正态分布函数


# 定义生成高斯核的函数
def gaussian_kernel(size, sigma=1.0):
    """
    生成高斯核。

    参数:
    size (int): 高斯核的大小。
    sigma (float): 高斯核的标准差。

    返回:
    numpy.ndarray: 高斯核。
    """
    kernel = np.fromfunction(
        lambda x, y: (1 / (2 * np.pi * sigma**2))
        * np.exp(-((x - size // 2) ** 2 + (y - size // 2) ** 2) / (2 * sigma**2)),
        (size, size),
    )
    return kernel / np.sum(kernel)  # 归一化核


# 定义绘制高斯核的函数
def plot_gaussian_kernel(kernel):
    """
    绘制和可视化高斯核的二维和三维图像。

    参数:
    kernel (numpy.ndarray): 要绘制的高斯核。
    """
    fig = plt.figure()  # 创建绘图对象

    # 绘制二维高斯核图像
    ax1 = fig.add_subplot(121)  # 添加子图位于左侧
    ax1.imshow(kernel, cmap="viridis", interpolation="none")  # 显示高斯核，使用viridis颜色映射
    # 在图像上标注每个像素的值
    for i in range(kernel.shape[0]):
        for j in range(kernel.shape[1]):
            ax1.text(j, i, f"{kernel[i, j]:.2f}", ha="center", va="center", color="r")
    ax1.set_title("2D Gaussian Kernel")  # 设置图像标题

    # 绘制三维高斯核图像
    ax2 = fig.add_subplot(122, projection="3d")  # 添加子图位于右侧，并设置为3D模式
    x, y = np.arange(0, kernel.shape[0], 1), np.arange(
        0, kernel.shape[1], 1
    )  # 创建x和y坐标网格
    x, y = np.meshgrid(x, y)
    ax2.plot_surface(x, y, kernel, cmap="viridis")  # 绘制高斯核的3D表面图
    ax2.set_title("3D Gaussian Kernel")  # 设置图像标题

    plt.show()  # 展示绘制的图像


kernel_size = 5  # 定义核的大小
sigma = 1.0  # 定义高斯核的标准差

# 生成高斯滤波器
kernel = gaussian_kernel(kernel_size, sigma)

# 绘制高斯滤波器的二维图像和三维图像
plot_gaussian_kernel(kernel)

# 边缘检测

In [ ]:
# 这段代码演示了如何使用 OpenCV 进行边缘检测。
# 它首先读取一幅图像，然后应用 Canny 算子来检测图像中的边缘。
# 最后，使用 Matplotlib 显示原始图像和检测到的边缘。

# 导入所需的库
import cv2  # OpenCV库，用于图像处理
import matplotlib.pyplot as plt  # Matplotlib库，用于显示图像

# 读取图像
# 'cat.png'是图像文件的路径，cv2.IMREAD_GRAYSCALE表示以灰度模式读取图像
image = cv2.imread("cat.png", cv2.IMREAD_GRAYSCALE)

# 使用Canny边缘检测算法检测图像边缘
# cv2.Canny函数接收图像和两个阈值（低阈值和高阈值），这里设置为50和150
edges = cv2.Canny(image, 50, 150)

# 使用Matplotlib显示结果
plt.figure(figsize=(8, 4))  # 设置图像显示大小

# 显示原始图像
plt.subplot(1, 2, 1)  # 创建一个1行2列的子图，并定位到第一个
plt.imshow(image, cmap="gray")  # 以灰度图模式显示原始图像
plt.title("Original Image")  # 设置子图的标题

# 显示使用Canny算法检测到的边缘
plt.subplot(1, 2, 2)  # 定位到第二个子图
plt.imshow(edges, cmap="gray")  # 以灰度图模式显示边缘图像
plt.title("Canny Edges")  # 设置子图的标题

plt.show()  # 显示所有子图

## 多种边缘检测算子效果比较

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# 读取图像
image = cv2.imread("cat.png", cv2.IMREAD_GRAYSCALE)

# ========== 1. Canny边缘检测 ==========
edges_canny = cv2.Canny(image, 50, 150)

# ========== 2. Sobel算子 ==========
# Sobel X方向梯度
sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
# Sobel Y方向梯度
sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
# Sobel总梯度（X和Y方向结合）
sobel_combined = cv2.magnitude(sobel_x, sobel_y)
sobel_combined = np.uint8(np.clip(sobel_combined, 0, 255))

# ========== 3. Laplacian算子 ==========
laplacian = cv2.Laplacian(image, cv2.CV_64F, ksize=3)
laplacian = np.uint8(np.clip(np.abs(laplacian), 0, 255))

# ========== 4. Prewitt算子 ==========
# 手动定义Prewitt算子核
kernel_prewitt_x = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
kernel_prewitt_y = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])

prewitt_x = cv2.filter2D(image, cv2.CV_64F, kernel_prewitt_x)
prewitt_y = cv2.filter2D(image, cv2.CV_64F, kernel_prewitt_y)
prewitt_combined = cv2.magnitude(prewitt_x, prewitt_y)
prewitt_combined = np.uint8(np.clip(prewitt_combined, 0, 255))

# ========== 5. Roberts算子 ==========
# 手动定义Roberts算子核
kernel_roberts_x = np.array([[1, 0], [0, -1]], dtype=np.float32)
kernel_roberts_y = np.array([[0, 1], [-1, 0]], dtype=np.float32)

roberts_x = cv2.filter2D(image, cv2.CV_64F, kernel_roberts_x)
roberts_y = cv2.filter2D(image, cv2.CV_64F, kernel_roberts_y)
roberts_combined = cv2.magnitude(roberts_x, roberts_y)
roberts_combined = np.uint8(np.clip(roberts_combined, 0, 255))

# ========== 6. Scharr算子（增强版Sobel） ==========
scharr_x = cv2.Scharr(image, cv2.CV_64F, 1, 0)
scharr_y = cv2.Scharr(image, cv2.CV_64F, 0, 1)
scharr_combined = cv2.magnitude(scharr_x, scharr_y)
scharr_combined = np.uint8(np.clip(scharr_combined, 0, 255))

# ========== 可视化结果 ==========
plt.figure(figsize=(16, 10))

# 原始图像
plt.subplot(3, 4, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Image')
plt.axis('off')

# Canny边缘检测
plt.subplot(3, 4, 2)
plt.imshow(edges_canny, cmap='gray')
plt.title('Canny Edge Detection')
plt.axis('off')

# Sobel算子
plt.subplot(3, 4, 3)
plt.imshow(sobel_combined, cmap='gray')
plt.title('Sobel Operator')
plt.axis('off')

# Laplacian算子
plt.subplot(3, 4, 4)
plt.imshow(laplacian, cmap='gray')
plt.title('Laplacian Operator')
plt.axis('off')

# Prewitt算子
plt.subplot(3, 4, 5)
plt.imshow(prewitt_combined, cmap='gray')
plt.title('Prewitt Operator')
plt.axis('off')

# Roberts算子
plt.subplot(3, 4, 6)
plt.imshow(roberts_combined, cmap='gray')
plt.title('Roberts Operator')
plt.axis('off')

# Scharr算子
plt.subplot(3, 4, 7)
plt.imshow(scharr_combined, cmap='gray')
plt.title('Scharr Operator')
plt.axis('off')

# Sobel X方向
plt.subplot(3, 4, 8)
plt.imshow(np.abs(sobel_x), cmap='gray')
plt.title('Sobel X Direction')
plt.axis('off')

# Sobel Y方向
plt.subplot(3, 4, 9)
plt.imshow(np.abs(sobel_y), cmap='gray')
plt.title('Sobel Y Direction')
plt.axis('off')

# Prewitt X方向
plt.subplot(3, 4, 10)
plt.imshow(np.abs(prewitt_x), cmap='gray')
plt.title('Prewitt X Direction')
plt.axis('off')

# Prewitt Y方向
plt.subplot(3, 4, 11)
plt.imshow(np.abs(prewitt_y), cmap='gray')
plt.title('Prewitt Y Direction')
plt.axis('off')

# 阈值化Sobel（二值化显示）
plt.subplot(3, 4, 12)
_, sobel_thresh = cv2.threshold(sobel_combined, 50, 255, cv2.THRESH_BINARY)
plt.imshow(sobel_thresh, cmap='gray')
plt.title('Sobel with Threshold')
plt.axis('off')

plt.tight_layout()
plt.show()

# ========== 各算子特性对比 ==========
print("=" * 60)
print("边缘检测算子特性对比：")
print("=" * 60)
print("1. Canny算子：")
print("   - 优点：双阈值检测，边缘连接好，抗噪声能力强")
print("   - 缺点：计算复杂度较高，需要调参")
print()

print("2. Sobel算子：")
print("   - 优点：计算简单，对噪声有一定抑制")
print("   - 缺点：边缘定位不够精确，易受噪声影响")
print()

print("3. Laplacian算子：")
print("   - 优点：各向同性，对边缘方向不敏感")
print("   - 缺点：对噪声敏感，易产生双边缘")
print()

print("4. Prewitt算子：")
print("   - 优点：实现简单，对垂直/水平边缘敏感")
print("   - 缺点：对斜边缘效果较差")
print()

print("5. Roberts算子：")
print("   - 优点：计算量小，对45°方向边缘敏感")
print("   - 缺点：对噪声敏感，边缘定位不准")
print()

print("6. Scharr算子：")
print("   - 优点：Sobel的改进版，旋转不变性更好")
print("   - 缺点：计算量稍大")
print("=" * 60)

# ========== 参数调优对比 ==========
# 使用不同参数的Canny算法对比
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 不同阈值组合
canny_params = [
    (30, 100, "Low Threshold"),
    (50, 150, "Medium Threshold"),
    (100, 200, "High Threshold"),
    (30, 200, "Wide Range"),
    (70, 120, "Narrow Range"),
    (50, 150, "Default (for reference)")
]

for idx, (low, high, title) in enumerate(canny_params[:5]):
    edges = cv2.Canny(image, low, high)
    row, col = divmod(idx, 3)
    axes[row, col].imshow(edges, cmap='gray')
    axes[row, col].set_title(f'{title}\n({low}, {high})')
    axes[row, col].axis('off')

# 原始图像作为对比
axes[1, 2].imshow(image, cmap='gray')
axes[1, 2].set_title('Original Image')
axes[1, 2].axis('off')

plt.suptitle('Canny Edge Detection with Different Parameters', fontsize=14)
plt.tight_layout()
plt.show()

## 算子比较
1. Canny算子
特点：最常用，提供完整的边缘检测流程（高斯滤波→梯度计算→非极大值抑制→双阈值检测）

效果：边缘连续性好，噪声抑制强

最佳用途：通用边缘检测，特别是需要清晰连续边缘的场景

2. Sobel算子
特点：一阶微分算子，分别检测水平和垂直边缘

效果：边缘较粗，对噪声有一定抵抗

最佳用途：需要梯度方向的边缘检测

3. Laplacian算子
特点：二阶微分算子，检测边缘的过零点

效果：对噪声敏感，会产生双边缘

最佳用途：边缘精确定位（需配合去噪）

4. Prewitt算子
特点：类似Sobel但权值更简单

效果：对水平和垂直边缘响应强

最佳用途：简单的边缘检测任务

5. Roberts算子
特点：最简单的2×2算子，计算量最小

效果：对45°方向边缘敏感

最佳用途：实时系统或计算资源有限的场景

6. Scharr算子
特点：Sobel的改进版，旋转不变性更好

效果：边缘定位更准确

最佳用途：需要更高精度梯度估计的场景

使用建议：
日常应用：首选Canny，效果最稳定

实时处理：考虑Sobel或Roberts，计算速度快

学术研究：可对比多种算子效果

资源受限：使用Prewitt或Roberts

精度要求高：使用Scharr或Canny+参数调优

# 图像分割 -- 大津算法 （Otsu's method）

## 核心问题，是找到一个阈值来将图像分割为二值图像（黑白图像）：前景和背景。
为了确定这个阈值1，大津算法采用遍历的方法完成:
- 首先假设阈值为0此时用阈值0将图像中所有像素划分为两类，并且计算这两类像素的类间方差,记E0;
- 然后继续假设阈值为1，同样可以将图像以阈值1划分为两类，此时可以得到类间方差E1，
- 依此类推便可以得到类间方差E0到E255;
- 最后遍历E0到E255这256个数值，得到**最大的类间方差Em**，则具有最大化类间方差效果的阈值为t=m。
- 利用确定好的阈值m来对图像进行分割了。

## 大津算法的完整实现步骤为:
1. 计算图像的直方图统计图像中每个灰度值的像素点个数。
2. 计算累积分布函数计算每个灰度值下的像素点个数占总像素点个数的比例。
3. 遍历所有可能的阈值t。对于每个阈值，根据累积分布函数计算前景和背景的像素比例以及平均灰度值，依据类间方差的公式计算类间方差。
4. 选择最大类间方差的阈值作为最佳阈值,这里假设最佳阈值为 m。
5. 使用最佳阈值对图像进行二值化处理,将灰度值小于等于m的像素设置为背景(通常将对应的灰度值设为0)，将灰度值大于m的像素设置为前景(常将对应的灰度值设为 255)。

In [ ]:
# 这段代码展示了如何使用 OpenCV 库进行图像分割。
# 它首先读取一幅灰度图像，然后使用大津算法（Otsu's method）自动找到一个阈值来将图像分割为二值图像（黑白图像）。
# 最后，使用 Matplotlib 将原始图像和分割后的图像显示出来。

# 导入所需的库
import cv2  # OpenCV库，用于图像处理
import numpy as np  # NumPy库，用于数值计算
import matplotlib.pyplot as plt  # Matplotlib库，用于显示图像

# 读取图像
# 'luna.jfif'是图像文件的路径，cv2.IMREAD_GRAYSCALE表示以灰度模式读取图像
image = cv2.imread("luna.jfif", cv2.IMREAD_GRAYSCALE)

# 使用大津算法（Otsu's method）自动找到最佳阈值进行图像二值化
# cv2.threshold函数返回两个值，第一个是找到的阈值，第二个是阈值化后的图像
# cv2.THRESH_BINARY是二值化类型，cv2.THRESH_OTSU是使用Otsu算法
_, thresholded = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 使用Matplotlib显示原始图像和分割后的图像
# 显示原始图像
plt.subplot(121), plt.imshow(image, cmap="gray")  # 在1行2列的子图中的第一个位置显示原始图像
plt.title("Original Image"), plt.xticks([]), plt.yticks([])  # 设置标题和去除坐标轴

# 显示分割后的图像
plt.subplot(122), plt.imshow(thresholded, cmap="gray")  # 在第二个位置显示分割后的图像
plt.title("Segmented Image"), plt.xticks([]), plt.yticks([])  # 设置标题和去除坐标轴

plt.show()  # 显示所有子图

In [ ]:
# 导入所需的库
import cv2  # OpenCV库，用于图像处理
import numpy as np  # NumPy库，用于数值计算
import matplotlib.pyplot as plt  # Matplotlib库，用于显示图像

# 读取图像
image = cv2.imread("luna.jfif", cv2.IMREAD_GRAYSCALE)

# 使用大津算法（Otsu's method）自动找到最佳阈值进行图像二值化
_, thresholded = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# ========== 补充代码：分别提取前景和背景 ==========

# 获取Otsu算法找到的阈值
ret, _ = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f"Otsu算法找到的最佳阈值: {ret}")

# 方法1：使用二值化图像作为掩码提取前景和背景
# 前景提取：前景像素保持原值，背景设为黑色(0)
foreground = cv2.bitwise_and(image, image, mask=thresholded)

# 背景提取：反转掩码，然后提取背景
background_mask = cv2.bitwise_not(thresholded)
background = cv2.bitwise_and(image, image, mask=background_mask)

# 方法2：创建纯色背景的合成图像
# 创建白色背景的前景图像
foreground_white_bg = foreground.copy()
foreground_white_bg[thresholded == 0] = 255  # 背景设为白色

# 创建黑色背景的前景图像（更清晰显示前景）
foreground_black_bg = foreground.copy()

# 创建彩色标记版本（可视化效果更好）
# 首先将灰度图转为BGR彩色图
image_colored = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

# 创建前景的红色标记版本
foreground_colored = image_colored.copy()
foreground_colored[thresholded == 255] = [0, 0, 255]  # 前景标记为红色

# 创建背景的绿色标记版本
background_colored = image_colored.copy()
background_colored[thresholded == 0] = [0, 255, 0]  # 背景标记为绿色

# 创建分离的前景和背景显示
# 纯前景：前景保持原灰度，背景为黑色
pure_foreground = np.zeros_like(image)
pure_foreground[thresholded == 255] = image[thresholded == 255]

# 纯背景：背景保持原灰度，前景为黑色
pure_background = np.zeros_like(image)
pure_background[thresholded == 0] = image[thresholded == 0]

# ========== 显示所有结果 ==========

plt.figure(figsize=(16, 10))

# 1. 原始图像和分割结果
plt.subplot(3, 4, 1)
plt.imshow(image, cmap='gray')
plt.title('1. Original Image')
plt.axis('off')

plt.subplot(3, 4, 2)
plt.imshow(thresholded, cmap='gray')
plt.title('2. Binary Segmentation\n(White=Foreground, Black=Background)')
plt.axis('off')

# 2. 分离的前景
plt.subplot(3, 4, 3)
plt.imshow(pure_foreground, cmap='gray')
plt.title('3. Pure Foreground\n(Background=Black)')
plt.axis('off')

plt.subplot(3, 4, 4)
plt.imshow(foreground_white_bg, cmap='gray')
plt.title('4. Foreground on White\n(Background=White)')
plt.axis('off')

# 3. 分离的背景
plt.subplot(3, 4, 5)
plt.imshow(pure_background, cmap='gray')
plt.title('5. Pure Background\n(Foreground=Black)')
plt.axis('off')

plt.subplot(3, 4, 6)
plt.imshow(background, cmap='gray')
plt.title('6. Background Only\n(Foreground=Black)')
plt.axis('off')

# 4. 彩色标记版本
plt.subplot(3, 4, 7)
plt.imshow(cv2.cvtColor(foreground_colored, cv2.COLOR_BGR2RGB))
plt.title('7. Foreground Marked Red')
plt.axis('off')

plt.subplot(3, 4, 8)
plt.imshow(cv2.cvtColor(background_colored, cv2.COLOR_BGR2RGB))
plt.title('8. Background Marked Green')
plt.axis('off')

# 5. 统计信息显示
plt.subplot(3, 4, 9)
# 创建一个简单的统计信息图
foreground_pixels = np.sum(thresholded == 255)
background_pixels = np.sum(thresholded == 0)
total_pixels = foreground_pixels + background_pixels

categories = ['Foreground', 'Background']
counts = [foreground_pixels, background_pixels]
percentages = [foreground_pixels/total_pixels*100, background_pixels/total_pixels*100]

bars = plt.bar(categories, counts, color=['red', 'green'])
plt.title('9. Pixel Distribution')
plt.ylabel('Number of Pixels')

# 在柱状图上显示百分比
for bar, percentage in zip(bars, percentages):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{percentage:.1f}%', ha='center', va='bottom')

# 6. 前景和背景直方图对比
plt.subplot(3, 4, 10)
foreground_values = image[thresholded == 255]
background_values = image[thresholded == 0]

plt.hist(foreground_values.flatten(), bins=50, alpha=0.7, color='red', label='Foreground')
plt.hist(background_values.flatten(), bins=50, alpha=0.7, color='green', label='Background')
plt.title('10. Intensity Histogram')
plt.xlabel('Pixel Intensity')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)

# 7. 合并显示（半透明覆盖）
plt.subplot(3, 4, 11)
plt.imshow(image, cmap='gray', alpha=0.7)
# 叠加前景区域
foreground_overlay = np.zeros_like(image)
foreground_overlay[thresholded == 255] = 255
plt.imshow(foreground_overlay, cmap='Reds', alpha=0.3)
plt.title('11. Overlay Visualization')
plt.axis('off')

# 8. 边界轮廓显示
plt.subplot(3, 4, 12)
contours, _ = cv2.findContours(thresholded, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contour_image = image.copy()
contour_image = cv2.cvtColor(contour_image, cv2.COLOR_GRAY2BGR)
cv2.drawContours(contour_image, contours, -1, (0, 255, 0), 2)
plt.imshow(cv2.cvtColor(contour_image, cv2.COLOR_BGR2RGB))
plt.title('12. Contour Detection')
plt.axis('off')

plt.suptitle(f'Image Segmentation Results (Otsu Threshold: {ret:.1f})', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# ========== 简洁版本：只显示主要结果 ==========
print("\n" + "="*60)
print("图像分割统计信息：")
print("="*60)
print(f"图像尺寸: {image.shape[1]} x {image.shape[0]} 像素")
print(f"总像素数: {total_pixels}")
print(f"前景像素数: {foreground_pixels} ({foreground_pixels/total_pixels*100:.1f}%)")
print(f"背景像素数: {background_pixels} ({background_pixels/total_pixels*100:.1f}%)")
print(f"Otsu最佳阈值: {ret}")
print("前景平均亮度: {:.1f}".format(np.mean(foreground_values) if len(foreground_values) > 0 else 0))
print("背景平均亮度: {:.1f}".format(np.mean(background_values) if len(background_values) > 0 else 0))
print("="*60)

# 简洁显示版本（如果只需要基本的前景/背景分离）
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(pure_foreground, cmap='gray')
plt.title('Foreground Only')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(pure_background, cmap='gray')
plt.title('Background Only')
plt.axis('off')

plt.suptitle('Basic Foreground/Background Separation', fontsize=14)
plt.tight_layout()
plt.show()